# Few-shot LLM classification: Llama

В этом ноутбуке проверяется few-shot подход к классификации новостей по рубрикам с помощью модели `Meta-Llama-3.1-8B-Instruct`.

Эксперимент повторяет схему few-shot классификации из предыдущих LLM-ноутбуков: модель получает список допустимых рубрик и по 2 фиксированных примера на каждый класс. Цель — проверить, улучшает ли few-shot prompting качество Llama относительно zero-shot и как Llama сравнивается с Gemma и Qwen.

In [1]:
import pandas as pd
import numpy as np
import os
import gc
import transformers
import torch
import logging
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
from tqdm.notebook import tqdm

logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
tqdm.pandas()

In [2]:
CATEGORIES = ["Мир", "Россия", "Экономика", "Наука и техника", "Спорт", "Культура"]
CATEGORIES_STR = ", ".join(CATEGORIES)

N_SHOTS    = 2    # примеров на класс
EXAMPLE_LEN = 300  # символов в каждом примере

DATA_PATH   = Path("news_data/cleaned_news_for_model.parquet")
SAMPLE_PATH = Path("news_data/llm_sample_1200.parquet")
REPORT_PATH = Path("reports/llm_fewshot_results.csv")
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

## Настройки few-shot эксперимента

Для каждой из 6 рубрик используется по 2 примера правильной классификации. Каждый пример ограничен первыми 300 символами текста, чтобы prompt оставался компактным.

Всего в prompt добавляется 12 fixed few-shot examples. Эти настройки совпадают с few-shot экспериментами для Gemma и Qwen, что позволяет сравнивать модели в одинаковых условиях.

In [3]:
# Загружаем ту же выборку что использовалась в zero-shot
df_sample = pd.read_parquet(SAMPLE_PATH)
print(f"Тестовая выборка: {len(df_sample)} строк")

# Для примеров берём всё что НЕ попало в выборку
df = pd.read_parquet(DATA_PATH)
df_model = df[["title", "text", "category_raw"]].dropna().copy()
df_model["llm_text"] = df_model["text"].astype(str)
df_model = df_model[df_model["llm_text"].str.len() > 0]

df_pool = df_model[~df_model.index.isin(df_sample.index)]
print(f"Пул для примеров: {len(df_pool)} строк")

Тестовая выборка: 1200 строк
Пул для примеров: 145419 строк


### Промежуточный вывод

Для эксперимента используется та же стратифицированная выборка из 1200 новостей, что и в zero-shot, Gemma few-shot и Qwen few-shot экспериментах. Это делает сравнение LLM-подходов корректным.

Few-shot примеры берутся из остального корпуса, который не входит в тестовую выборку. Это снижает риск утечки данных: модель не получает в prompt сами тестовые объекты.

In [4]:
few_shot_examples = []
for cat in CATEGORIES:
    cat_examples = (
        df_pool[df_pool["category_raw"] == cat]
        .sample(n=N_SHOTS, random_state=42)
    )
    for _, row in cat_examples.iterrows():
        few_shot_examples.append({
            "category": cat,
            "text": row["llm_text"][:EXAMPLE_LEN],
        })

# Проверяем что примеры не пересекаются с тестовой выборкой по индексу
example_idx = {ex["text"] for ex in few_shot_examples}
leaks = df_sample["llm_text"].apply(lambda t: t[:EXAMPLE_LEN] in example_idx).sum()
assert leaks == 0, f"Утечка! {leaks} примеров из промпта есть в выборке"

print(f"Few-shot примеров: {len(few_shot_examples)} ({N_SHOTS} на класс × {len(CATEGORIES)} классов)")
print(f"Каждый пример — первые {EXAMPLE_LEN} символов")

# Показываем что получилось
for ex in few_shot_examples:
    print(f"\n[{ex['category']}] {ex['text'][:80]}...")

Few-shot примеров: 12 (2 на класс × 6 классов)
Каждый пример — первые 300 символов

[Мир] Politico: ЕС разрабатывает план по частичному членству Украины в 2027 году
Семен...

[Мир] Sky News: Лидеры ЕС могут поехать в США, чтобы повлиять на Трампа по Украине
Вик...

[Россия] Сенатор Косачев: Превращение БРИКС или ШОС в военный блок бесперспективно
Идеи п...

[Россия] Врачи НИИ Склифосовского провели уникальную трансплантацию кисти от донора
Росси...

[Экономика] Тайфун образовался к югу от Японии, он может затронуть Курильские острова
Тайфун...

[Экономика] Депутат Госдумы Нилов: Цены на цветы следует ограничить законодательно
Фото: Ser...

[Наука и техника] Daily Mail: Распятие Иисуса Христа могло произойти 3 апреля 33 года нашей эры
Ек...

[Наука и техника] eBioMedicine: У пациентов с длительным COVID повышается уровень тау-белка
Екатер...

[Спорт] Вратарь сборной России по водному поло Федотов: Были готовы играть с Украиной
Фо...

[Спорт] Лыжник Коростелев выполнил олимпийский нормат

### Промежуточный вывод

Сформировано 12 few-shot примеров: по 2 примера на каждую из 6 рубрик. Примеры фиксируются через `random_state=42`, поэтому эксперимент воспроизводим.

Важно учитывать, что примеры выбраны случайно, а не подобраны вручную или по близости к классифицируемому тексту. Поэтому это fixed few-shot подход, а не retrieval-based few-shot.

Некоторые few-shot примеры могут выглядеть неоднозначно для человека, потому что используются исходные редакционные рубрики Lenta.ru. Это может влиять на качество LLM-классификации.

In [5]:
SYSTEM_PROMPT_FS = (
    f"Ты классификатор новостных статей.\n"
    f"Определи категорию текста и ответь ТОЛЬКО одним из вариантов: {CATEGORIES_STR}.\n"
    f"Никаких пояснений — только одно слово или фраза из списка."
)

def build_few_shot_messages(text):
    """Собирает messages: system + примеры как диалог + целевой текст."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT_FS}]

    # Примеры — диалог где модель уже "отвечала" правильно
    for ex in few_shot_examples:
        messages.append({"role": "user",      "content": ex["text"]})
        messages.append({"role": "assistant", "content": ex["category"]})

    # Целевой текст — последний вопрос без ответа
    messages.append({"role": "user", "content": text})
    return messages

def classify_few_shot(text, pipeline, max_new_tokens=20):
    messages = build_few_shot_messages(text)
    output = pipeline(messages, max_new_tokens=max_new_tokens, do_sample=False)
    response = output[0]["generated_text"][-1]["content"].strip()

    for cat in CATEGORIES:
        if cat.lower() in response.lower():
            return cat

    print(f"[Неизвестно] Ответ модели: '{response}' | Текст: '{text[:800]}'")
    return "Неизвестно"

def free_memory(obj):
    del obj
    gc.collect()
    torch.mps.empty_cache()
    print("Память освобождена")

def compute_metrics(y_true, y_pred, model_name):
    mask = y_pred != "Неизвестно"
    accuracy    = accuracy_score(y_true[mask], y_pred[mask])
    macro_f1    = f1_score(y_true[mask], y_pred[mask], average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_true[mask], y_pred[mask], average="weighted", zero_division=0)
    unknown_rate = (~mask).mean()

    print(f"\n=== {model_name} ===")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print(f"Неизвестно:  {unknown_rate:.1%} ответов не распознано")
    print(classification_report(y_true[mask], y_pred[mask], zero_division=0))

    return accuracy, macro_f1, weighted_f1, unknown_rate

## Few-shot prompt и оценка качества

Prompt строится как диалог: сначала системная инструкция, затем пары `текст новости → правильная рубрика`, затем целевой текст для классификации.

Модель должна вернуть только одну из 6 допустимых рубрик. После генерации ответ парсится и сопоставляется со списком классов. Если рубрика не найдена, ответ помечается как `Неизвестно`.

Качество оценивается по Accuracy, Macro F1 и Weighted F1. Доля ответов `Неизвестно` выводится отдельно.

In [6]:
# Посмотреть как выглядит промпт перед запуском
test_messages = build_few_shot_messages("Тестовый текст новости")
for msg in test_messages:
    role = msg["role"].upper()
    preview = msg["content"][:100].replace("\n", " ")
    print(f"[{role}] {preview}...")
    print()

[SYSTEM] Ты классификатор новостных статей. Определи категорию текста и ответь ТОЛЬКО одним из вариантов: Мир...

[USER] Politico: ЕС разрабатывает план по частичному членству Украины в 2027 году Семен Александров (старши...

[ASSISTANT] Мир...

[USER] Sky News: Лидеры ЕС могут поехать в США, чтобы повлиять на Трампа по Украине Виктория Кондратьева (Р...

[ASSISTANT] Мир...

[USER] Сенатор Косачев: Превращение БРИКС или ШОС в военный блок бесперспективно Идеи превратить БРИКС или ...

[ASSISTANT] Россия...

[USER] Врачи НИИ Склифосовского провели уникальную трансплантацию кисти от донора Российские врачи НИИ скор...

[ASSISTANT] Россия...

[USER] Тайфун образовался к югу от Японии, он может затронуть Курильские острова Тайфун «Нари», пятый в это...

[ASSISTANT] Экономика...

[USER] Депутат Госдумы Нилов: Цены на цветы следует ограничить законодательно Фото: Sergey Elagin / Busines...

[ASSISTANT] Экономика...

[USER] Daily Mail: Распятие Иисуса Христа могло произойти 3 апреля 33 года н

### Промежуточный вывод

Проверка prompt показывает, что Llama получает тот же набор few-shot примеров, что и Gemma и Qwen. Это важно для честного сравнения: различия в качестве должны быть связаны преимущественно с моделью, а не с разными примерами или разным форматом prompt.

In [7]:
pipeline_llama = transformers.pipeline(
    "text-generation",
    model="meta-llama/Meta-Llama-3.1-8B-Instruct",
    dtype="auto",
    device_map="auto",
)
print("Llama загружена")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Llama загружена


### Промежуточный вывод

Модель `Meta-Llama-3.1-8B-Instruct` успешно загружена и используется только в режиме inference. Fine-tuning модели не выполняется: классификация строится исключительно через few-shot prompting.

In [8]:
print(f"Few-shot классификация {len(df_sample)} текстов через Llama...")
df_sample["pred_llama_fs"] = df_sample["llm_text"].progress_apply(
    lambda text: classify_few_shot(text, pipeline_llama)
)
df_sample["pred_llama_fs"].value_counts()

Few-shot классификация 1200 текстов через Llama...


  0%|          | 0/1200 [00:00<?, ?it/s]

[Неизвестно] Ответ модели: 'Образование' | Текст: 'Сдавшей ЕГЭ на 400 баллов школьнице Яшмолкиной предложили стажировку в MAX
Варвара Кошечкина (редактор отдела оперативной информации)
Варвара Кошечкина (редактор отдела оперативной информации)
Фото: Alina Troeva / Shutterstock / Fotodom
Фото: Alina Troeva / Shutterstock / Fotodom
Выпускнице московской школы №1514 Надежде Яшмолкиной, сдавшей Единый государственный экзамен (ЕГЭ) в 2025 году на 400 баллов, предложили стажировку в команде цифровой платформы MAX. Об этом со ссылкой на VK сообщает РИА Новости . Уточняется, что школьница уже побывала в офисе компании. Там она познакомилась с экспертами и узнала о возможностях развиваться в IT-сфере с помощью проектов VK Education. Также во время визита сотрудники рассказали Яшмолкиной, как присоединились к команде VK после учебы на совместных с в'
[Неизвестно] Ответ модели: 'Среда обитания' | Текст: 'В Лондоне увидели белку с электронной сигаретой, и экологи забили тревогу
Нина Ташевская (Ред

pred_llama_fs
Мир                357
Россия             346
Экономика          216
Культура           109
Наука и техника     91
Спорт               76
Неизвестно           5
Name: count, dtype: int64

### Промежуточный вывод

Для всех 1200 новостей получены few-shot предсказания Llama. Далее качество оценивается теми же метриками, что и в остальных LLM-экспериментах: Accuracy, Macro F1 и Weighted F1.

Также отдельно фиксируется доля ответов `Неизвестно`, то есть случаев, когда ответ модели не удалось сопоставить с одной из допустимых рубрик.

In [9]:
free_memory(pipeline_llama)

Память освобождена


In [10]:
acc_l, mf1_l, wf1_l, unk_l = compute_metrics(
    df_sample["category_raw"], df_sample["pred_llama_fs"], "Llama-3.1-8B-Instruct few-shot"
)


=== Llama-3.1-8B-Instruct few-shot ===
Accuracy:    0.8192
Macro F1:    0.8274
Weighted F1: 0.8216
Неизвестно:  0.4% ответов не распознано
                 precision    recall  f1-score   support

       Культура       0.55      0.95      0.70        63
            Мир       0.85      0.81      0.83       371
Наука и техника       0.85      0.93      0.89        83
         Россия       0.79      0.81      0.80       338
          Спорт       0.97      0.91      0.94        81
      Экономика       0.89      0.74      0.81       259

       accuracy                           0.82      1195
      macro avg       0.82      0.86      0.83      1195
   weighted avg       0.83      0.82      0.82      1195



### Промежуточный вывод

Llama few-shot показала Accuracy ≈ 0.819, Macro F1 ≈ 0.827 и Weighted F1 ≈ 0.822. Это заметно лучше Llama zero-shot, где Accuracy была около 0.714.

Прирост по Accuracy составляет примерно 10.5 процентного пункта, что делает Llama одной из моделей, наиболее сильно выигравших от few-shot prompting. Однако результат всё равно ниже Qwen few-shot и значительно ниже TF-IDF + LinearSVC.

Llama few-shot лучше всего классифицирует рубрики `Спорт` и `Наука и техника`. Это показывает, что модель хорошо справляется с тематически более отделимыми классами.

Самый слабый F1-score у рубрики `Культура`: precision около 0.55 при recall около 0.95. Это означает, что модель часто предсказывает `Культура` для новостей из других классов. Похожий эффект наблюдался и у Qwen few-shot, поэтому часть ошибок может быть связана с особенностями fixed few-shot примеров.

In [11]:
results = pd.DataFrame([
    
    {
        "experiment": "15_llama3.1_8b_fewshot",
        "model": "meta-llama/Meta-Llama-3.1-8B-Instruct",
        "input": "text", "classifier": f"few-shot LLM ({N_SHOTS}/class)",
        "sample_size": len(df_sample),
        "accuracy": acc_l, "macro_f1": mf1_l,
        "weighted_f1": wf1_l, "unknown_rate": unk_l,
    },
])

results.to_csv(
    REPORT_PATH,
    mode="a",
    header=not REPORT_PATH.exists(),
    index=False,
)
results

,experiment,model,input,classifier,sample_size,accuracy,macro_f1,weighted_f1,unknown_rate
0,15_llama3.1_8b_fewshot,meta-llama/Meta-Llama-3.1-8B-Instruct,text,few-shot LLM (2/class),1200,0.819247,0.827445,0.821551,0.004167


### Промежуточный вывод

Результаты Llama few-shot сохранены в общий CSV-файл с LLM-экспериментами. Эти данные можно использовать для итогового сравнения zero-shot и few-shot подходов между Llama, Qwen и Gemma.

In [12]:
model_col = "pred_llama_fs"  

errors = df_sample[df_sample["category_raw"] != df_sample[model_col]][[
    "llm_text", "category_raw", model_col
]].rename(columns={"llm_text": "text", "category_raw": "true", model_col: "pred"})

errors["text_len"] = errors["text"].str.len()

print(f"Ошибок: {len(errors)} из {len(df_sample)}")
with pd.option_context("display.max_colwidth", 300):
    display(errors.head(20))

Ошибок: 221 из 1200


,text,true,pred
107059,"Фитнес-тренер упомянула еду после 18:00 и вред углеводов среди мифов о похудении\nФитнес-тренер Ксения Гуфранова назвала топ мифов о похудении. Об этом сообщают «Известия» . Самое распространенное заблуждение, по мнению специалиста, связано с ограничениями по времени. «Многие уверены, что ужины ...",Спорт,Культура
26262,"На Западном берегу Иордана избили палестинского режиссера Хамдана Баллала\nФото: Monika Skolimowska / Globallookpress\nФото: Monika Skolimowska / Globallookpress\nНа Западном берегу реки Иордан израильские поселенцы избили палестинского режиссера Хамдана Баллала, снявшего документальный фильм «Н...",Культура,Мир
140930,"Доцент Балынин: Шестидневных рабочих недель в России в 2026 году не будет\nВ 2026 году для жителей России, работающих по стандартному пятидневному графику, введение шестидневной рабочей недели не планируется. Об этом РИА Новости заявил доцент кафедры общественных финансов Финансового университет...",Россия,Экономика
50050,"Средства от продаж в День мороженого в ГУМе переведут фонду «Наука – детям»\nВ московском ГУМе, прошел традиционный День мороженого. Праздник мороженого прошел под названием «""Артек"" на все 100!». Дело в том, что легендарный международный детский центр отмечает в этом году юбилей. Этому событию ...",Россия,Культура
79049,Слюсарь: 2 сотрудника интерната в Ростовской области пострадали при атаке БПЛА\nЕвгений Силаев (Ночной линейный редактор)\nЕвгений Силаев (Ночной линейный редактор)\nФото: Alexander Legky / Global Look Press\nФото: Alexander Legky / Global Look Press\nУкраинский беспилотный летательный аппарат (...,Россия,Мир
120116,Премьер Канады Карни: Планов заключать соглашение о свободной торговле с КНР нет\nМарк Карни. Фото: Patrick Doyle / Keystone Press Agency / Global Look Press\nМарк Карни. Фото: Patrick Doyle / Keystone Press Agency / Global Look Press\nПланов заключать соглашение о свободной торговле с Китаем не...,Мир,Экономика
63266,Экс-народный губернатор Донбасса Губарев: Алаудинов устроил травлю Дивнича\nАпти Алаудинов. Фото: Petrov Sergey / news.ru / Globallookpress.com\nАпти Алаудинов. Фото: Petrov Sergey / news.ru / Globallookpress.com\nКомандир спецназа «Ахмат» и замглавы Главного военно-политического управления Мино...,Россия,Культура
75911,Машков: За два года «Театральный бульвар» стремительно завоевал любовь зрителей\nИдея «Театрального бульвара» перспективна для всех участников фестиваля. Об этом рассказалпредседатель Союза театральных деятелей РФ Владимир Машков . «Театральный бульвар» очень успешный проект Москвы . За эти два ...,Россия,Культура
69832,"Подольская заявила о праве россиян взять больничный и при отсутствии болезни\nЭксперт РАНХиГС Татьяна Подольская заявила, что россияне могут оформить больничный лист, даже если не болеют. Об этом она высказалась в беседе с РИА Новости . Специалист заявила о праве россиян взять больничный и при о...",Россия,Экономика
101681,Небензя: Украина скоро дойдет до «гитлерюгенда» ради пополнения ВСУ\nЕвгений Силаев (Ночной линейный редактор)\nЕвгений Силаев (Ночной линейный редактор)\nФото: Валерий Шарифулин / POOL / РИА Новости\nФото: Валерий Шарифулин / POOL / РИА Новости\nКиев рискует скоро дойти до «гитлерюгенда» ради п...,Мир,Россия


### Промежуточный вывод

Llama few-shot ошиблась на 221 объекте из 1200. Это почти столько же, сколько Gemma few-shot, но больше, чем Qwen few-shot.

Просмотр отдельных ошибок показывает, что часть новостей находится на границе нескольких рубрик. Особенно сложными остаются новости, где пересекаются российская, международная и экономическая повестка, а также случаи, где модель слишком широко использует рубрику `Культура`.

In [13]:
unknown = df_sample[df_sample[model_col] == "Неизвестно"][["llm_text", "category_raw", model_col]]\
    .rename(columns={"llm_text": "text", "category_raw": "true", model_col: "pred"})

print(f"Неизвестно: {len(unknown)}")
with pd.option_context("display.max_colwidth", 800):
    display(unknown)

Неизвестно: 5


,text,true,pred
57532,"Сдавшей ЕГЭ на 400 баллов школьнице Яшмолкиной предложили стажировку в MAX\nВарвара Кошечкина (редактор отдела оперативной информации)\nВарвара Кошечкина (редактор отдела оперативной информации)\nФото: Alina Troeva / Shutterstock / Fotodom\nФото: Alina Troeva / Shutterstock / Fotodom\nВыпускнице московской школы №1514 Надежде Яшмолкиной, сдавшей Единый государственный экзамен (ЕГЭ) в 2025 году на 400 баллов, предложили стажировку в команде цифровой платформы MAX. Об этом со ссылкой на VK сообщает РИА Новости . Уточняется, что школьница уже побывала в офисе компании. Там она познакомилась с экспертами и узнала о возможностях развиваться в IT-сфере с помощью проектов VK Education. Также во время визита сотрудники рассказали Яшмолкиной, как присоединились к команде VK после учебы на совме...",Россия,Неизвестно
137641,"В Лондоне увидели белку с электронной сигаретой, и экологи забили тревогу\nНина Ташевская (Редактор отдела «Среда обитания»)\nНина Ташевская (Редактор отдела «Среда обитания»)\nФото: Jennifer Tepp / Shutterstock / Fotodom\nФото: Jennifer Tepp / Shutterstock / Fotodom\nВ центре Лондона белка «закурила» вейп и привлекла внимание общественности к глобальной проблеме. Экологи забили тревогу из-за обострившегося кризиса утилизации отходов, пишет Metro. На снятых очевидцами кадрах видно, как животное держит передними лапами электронную сигарету и грызет ее мундштук. После того как видео завирусилось в соцсетях, Британское общество защиты животных (RSPCA) напомнило об опасности выброшенных отходов. Животные путают вейпы с едой — их может привлекать сладкий запах остатков жидкости. По словам с...",Экономика,Неизвестно
118022,"Роспотребнадзор: Нахождение в комнате с низкой влажностью опасно для глаз и носа\nНина Ташевская (Редактор отдела «Среда обитания»)\nНина Ташевская (Редактор отдела «Среда обитания»)\nФото: Svetlana Khutornaia / Shutterstock / Fotodom\nФото: Svetlana Khutornaia / Shutterstock / Fotodom\nДлительное нахождение в комнатах с низкой влажностью воздуха зимой может вызывать ощущение сухости в носоглотке, рези в глазах, снижение внимания и работоспособности. Об опасности таких условий в помещении предупредили россиян в пресс-службе Роспотребнадзора , пишет РИА Новости . По санитарным нормам температура воздуха в жилых комнатах зимой должна достигать плюс 18-24 градусов Цельсия в зависимости от региона. А оптимальная относительная влажность в помещении находится в интервале 30-45 процентов в хо...",Экономика,Неизвестно
87993,"Эколог Бурмистров: Летучая мышь может быть переносчиком насекомых-паразитов\nФото: Daniel Karmann / dpa / Globallookpress.com\nФото: Daniel Karmann / dpa / Globallookpress.com\nЛетучие мыши нередко залетают в квартиры россиян. Как себя вести с дикими животными, чтобы не навредить ни себе, ни им, телеканалу «Москва 24» рассказал начальник управления охраны и мониторинга объектов животного мира столичного Департамента природопользования и охраны окружающей среды Сергей Бурмистров . Как разъяснил специалист, летучие мыши помогают контролировать численность насекомых, поддерживают пищевые цепочки и служат своеобразным показателем экологического здоровья города. Сейчас большинство рукокрылых находятся в состоянии анабиоза, а оставшиеся готовятся к спячке. Для зимовки они выбирают укромные м...",Экономика,Неизвестно
72521,"Житель Оренбурга согласился заменить домофон и лишился 3 миллионов рублей\nНина Ташевская (Редактор отдела «Среда обитания»)\nНина Ташевская (Редактор отдела «Среда обитания»)\nЖитель Оренбурга попытался заменить домофон и лишился миллионов рублей. Об этом сообщает управление МВД по российскому региону. Неизвестные связались с 83-летним мужчиной и представились сотрудниками компании по замене домофонов. Россиянин поверил аферистам и согласился заменить устройство. Перед проведением процедуры пенсионера попросили продиктовать паспортные данные. После этого мужчине поступил новый звонок с сообщением, что его персональные данные скомпрометированы — под угро

### Промежуточный вывод

У Llama few-shot обнаружено 5 ответов `Неизвестно`, то есть около 0.4% выборки. Это небольшая доля, поэтому основное снижение качества связано не с парсингом ответа, а с выбором неверной рубрики.

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Llama zero-shot",
        "accuracy": 0.714,
        "macro_f1": 0.736,
        "weighted_f1": 0.703,
    },
    {
        "model": "Llama few-shot",
        "accuracy": acc_l,
        "macro_f1": mf1_l,
        "weighted_f1": wf1_l,
    },
])

for col in ["accuracy", "macro_f1", "weighted_f1"]:
    comparison[f"{col}_delta_pp"] = (
        comparison[col] - comparison.loc[0, col]
    ) * 100

comparison

### Промежуточный вывод

Few-shot prompting сильно улучшил Llama относительно zero-shot. Прирост по Accuracy составил около 10.5 процентного пункта, по Weighted F1 — около 11.9 процентного пункта.

## Ключевой вывод

Llama few-shot показала Accuracy ≈ 0.819, Macro F1 ≈ 0.827 и Weighted F1 ≈ 0.822 на той же стратифицированной выборке из 1200 новостей.

По сравнению с zero-shot режимом качество выросло примерно на 10.5 процентного пункта по Accuracy. Это показывает, что Llama хорошо использует few-shot примеры.

Однако среди few-shot LLM лучший результат остается у Qwen, а все LLM-подходы заметно уступают TF-IDF + LinearSVC. Поэтому основной вывод работы сохраняется: для данной задачи классический supervised ML на TF-IDF-признаках эффективнее prompting локальных LLM без fine-tuning.